In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px
import numpy as np
import re
pd.set_option('display.float_format', '{:.0f}'.format)




In [2]:
df = pd.read_excel(r'D:\walmart_USA_Terms.xlsx')
df


,Row ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,deuorichan,deuorichan_days,day_shiping,YearMonth
0,1,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Bookcases,Bush Somerset Collection Bookcase,262,2,0,42,3,3,3,2016-11
1,2,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",732,3,0,220,3,3,3,2016-11
2,3,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,Labels,Self-Adhesive Address Labels for Typewriters b...,15,2,0,7,4,4,4,2016-06
3,4,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Tables,Bretford CR4500 Series Slim Rectangular Table,958,5,0,-383,7,7,7,2015-10
4,5,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Storage,Eldon Fold 'N Roll Cart System,22,2,0,3,7,7,7,2015-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,7173,2017-03-26,2017-03-30,Standard Class,HK-14890,Heather Kirkland,Corporate,United States,Houston,Texas,...,Accessories,Sony 64GB Class 10 Micro SDHC R40 Memory Card,144,5,0,2,4,4,4,2017-03
9986,7174,2017-03-26,2017-03-30,Standard Class,HK-14890,Heather Kirkland,Corporate,United States,Houston,Texas,...,Copiers,Canon PC1080F Personal Copier,2400,5,0,570,4,4,4,2017-03
9987,7175,2017-03-26,2017-03-30,Standard Class,HK-14890,Heather Kirkland,Corporate,United States,Houston,Texas,...,Paper,Xerox 1951,74,3,0,23,4,4,4,2017-03
9988,7176,2017-03-26,2017-03-30,Standard Class,HK-14890,Heather Kirkland,Corporate,United States,Houston,Texas,...,Appliances,Belkin 5 Outlet SurgeMaster Power Centers,87,8,1,-227,4,4,4,2017-03


In [3]:
def get_mean_sales_by_ship_mode(df):
    """
    """
    return df.groupby("Ship Mode")["Sales"].mean()


def get_top_sales_sub_categories(df):
    """
    """
    return df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False).reset_index()


def get_worst_customers_sales(df, limit=10):
    """
    """
    return df.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False).reset_index().tail(limit)


def get_top_customers_profit(df, limit=10):
    """
    """
    return df.groupby("Customer Name")["Profit"].sum().sort_values(ascending=False).reset_index().head(limit)


def get_worst_sub_categories_profit(df, limit=3):
    """
    """
    return df.groupby("Sub-Category")["Profit"].sum().sort_values(ascending=False).reset_index().tail(limit)


def get_top_states_sales(df, limit=5):
    """
    """
    return df.groupby("State")["Sales"].sum().sort_values(ascending=False).reset_index().head(limit)


def get_state_sales_and_profit(df):
    """
    """
    return df.groupby("State")[["Sales", "Profit"]].sum().reset_index()


def get_loss_states(df):
    """
    """
    state_sales_profit = get_state_sales_and_profit(df)
    loss_states = state_sales_profit[state_sales_profit["Profit"] < 0]
    return loss_states


def get_profitable_states_sorted_by_sales(df):
    """
    """
    state_sales_profit = get_state_sales_and_profit(df)
    win_states = state_sales_profit[state_sales_profit["Profit"] >= 0]
    return win_states.sort_values(by="Sales", ascending=True)


def get_top_sales_dates(df, limit=10):
    """
    """
    return df.groupby("Order Date")["Sales"].sum().sort_values(ascending=False).reset_index().head(limit)


def get_profit_by_segment(df):
    """
    """
    return df.groupby("Segment")["Profit"].sum().sort_values(ascending=False).reset_index()


def get_profit_by_category(df):
    """
    """
    return df.groupby("Category")["Profit"].sum().sort_values(ascending=False).reset_index()


def calculate_discount_profit_correlation(df):
    """
    """
    return df["Discount"].corr(df["Profit"])


def get_sales_and_profit_by_discount(df):
    """
    """
    return df.groupby("Discount")[["Sales", "Profit"]].sum().reset_index()


def get_mean_profit_by_discount(df):
    """
    """
    return df.groupby("Discount")["Profit"].mean().reset_index()


def get_best_ship_mode_profit(df, limit=1):
    """
    """
    return df.groupby("Ship Mode")["Profit"].sum().reset_index().head(limit)


In [4]:
def analyze_input(text, df):
    text = text.lower()

    if "ship mode" in text:
        return get_mean_sales_by_ship_mode(df)

    elif "top customers profit" in text:
        return get_top_customers_profit(df)

    elif "loss states" in text:
        return get_loss_states(df)

    else:
        return "مش فاهم طلبك"

In [5]:
def chatbot(df):
    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            break

        result = analyze_input(user_input, df)
        print(result)

In [6]:
INTENTS = {
    "top_customers_profit": [
        "top customers profit",
        "best customers profit",
        "highest profit customers"
    ],
    "loss_states": [
        "loss states",
        "states with loss",
        "negative profit states"
    ],
    "ship_mode_sales": [
        "ship mode",
        "shipping sales",
        "sales by ship"
    ]
}


In [7]:
def analyze_input(text, df):
    text = text.lower()

    for intent, keywords in INTENTS.items():
        if any(k in text for k in keywords):

            if intent == "top_customers_profit":
                return get_top_customers_profit(df)

            elif intent == "loss_states":
                return get_loss_states(df)

            elif intent == "ship_mode_sales":
                return get_mean_sales_by_ship_mode(df)

    return "I didn't understand your request"

In [8]:
def chatbot(df):
    print("Bot is ready... type exit to stop")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: bye 👋")
            break

        result = analyze_input(user_input, df)
        print("Bot:", result)

In [9]:
def run_chat(df):
    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: bye 👋")
            break

        result = analyze_input(user_input, df)
        print("Bot:", result)

In [10]:
run_chat(df)

Bot: I didn't understand your request
Bot: I didn't understand your request
Bot: I didn't understand your request
Bot: Ship Mode
First Class      229
Same Day         236
Second Class     236
Standard Class   228
Name: Sales, dtype: float64
Bot: I didn't understand your request
Bot: bye 👋
